In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm 
import seaborn as sns
import math

from scipy.stats import gaussian_kde
from scipy.signal import argrelextrema

from scipy.sparse import issparse

import gc


# set figure size for better visibility
sc.settings.set_figure_params(dpi=100, frameon=False)
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)

In [2]:
## --- LOAD DATA ---
adata = sc.read_h5ad("../data/data_filtered_raw.h5ad")

## PERSON RESIDUALS NORMALIZATION

Using Pearson residuals (see paper https://doi.org/10.1186/s13059-019-1874-1)

Uncouple poisson noise from technical issues from biological heterogeneity: handles the "depth" scaling and variance stabilization.

Notes:

1. We do NOT filter genes yet. The method needs most genes to fit the model.
2. We perform this on the WHOLE dataset (all plates/conditions mixed).
3. Might be needed to still need some extra correction for batch effects (Harmony / scVI). We will see this with the UMAP and mESC control.

In [3]:
# --- DATA TYPE DIAGNOSTICS ---
# We verify if the matrix contains integers (counts) or floats (already normalized?)

if issparse(adata.X):
    data_sample = adata.X.data[:10] # Look at first 10 non-zero elements
else:
    data_sample = adata.X[:10, :10].flatten()

print("--- MATRIX INSPECTION ---")
print(f"Data Sample: {data_sample}")
print(f"Data Type: {adata.X.dtype}")

print(f"Range of values of loaded data: {adata.X.min():.2f} to {adata.X.max():.2f}")

# CHECK 1: Are they already normalized? (Small numbers < 1)
if np.any((data_sample > 0) & (data_sample < 1)):
    print("STOP: Pearson Residuals require RAW INTEGER COUNTS.")

# CHECK 2: Are they floats looking like integers? (e.g. 5.0)
# We calculate the sum of fractional parts. Should be 0.
if issparse(adata.X):
    fractional_part = np.sum(adata.X.data % 1)
else:
    fractional_part = np.sum(adata.X % 1)

if fractional_part == 0:
    print("Check Passed: Data represents integer counts.")
    
    # Force casting to integer to satisfy the algorithm's strict requirements
    # This changes 5.0 to 5, silencing the warning.
    if adata.X.dtype != int:
        print("Casting matrix from Float to Int...")
        adata.X = adata.X.astype(int)
else:
    print("WARNING: Rounding data to nearest integer.")
    # we round to nearest integer if we trust the raw nature
    adata.X = np.rint(adata.X).astype(int)

# --- SAVE RAW STATE (Backup) ---
# Save the raw counts in 'counts' layer.
adata.layers["counts"] = adata.X.copy()
print("Raw counts saved to adata.layers['counts'].")

# --- NORMALIZATION (Hafemeister & Satija / Pearson Residuals) ---
print("\n--- RUNNING NORMALIZATION ---")
sc.experimental.pp.normalize_pearson_residuals(
    adata, 
    theta=100,      # Overdispersion parameter (100 is standard for UMI data)
    clip=True,      # Clips extreme outliers (√N) to stabilize variance
    check_values=True,
    layer="counts"  # Explicitly tell it to use the raw counts layer we just saved
)

print("Normalization Complete.")
print(f"Range of new values: {adata.X.min():.2f} to {adata.X.max():.2f}")

# Save normalized data
adata.write("../data/data_pearson_norm.h5ad", compression="gzip")

--- MATRIX INSPECTION ---
Data Sample: [1.   0.05 3.   0.12 1.   3.   1.   0.33 1.   1.  ]
Data Type: float32
Range of values of loaded data: 0.00 to 13207.51
STOP: Pearson Residuals require RAW INTEGER COUNTS.
Raw counts saved to adata.layers['counts'].

--- RUNNING NORMALIZATION ---
computing analytic Pearson residuals on counts


/Users/jaime/miniforge3/envs/lncRNA_ASO/lib/python3.10/site-packages/scanpy/experimental/pp/_normalization.py:76: RuntimeWarning: invalid value encountered in divide
  residuals = diff / np.sqrt(mu + mu**2 / theta)


    finished (0:00:28)
Normalization Complete.
Range of new values: 0.00 to 13208.00
